# Numerical libraries for PDEs

Given the enormous complexity that PDE can present, there are scientific libraries targeted to their solutions. Furhtermore, those libraries implement more advanced methods like the Finite Element Method. Here we will show just a few  to highlight their use. In particular, two libraries that use the powerful [finite element method](https://en.wikipedia.org/wiki/Finite_element_method?useskin=vector) to solve PDE in complex geometries, will be shown at the end. 

In [ ]:
# Import needed utils, like for embeding videos and url
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'utils')))
from embed import embed


In [ ]:
embed("https://visualpde.com/explore")

## Finite differences


### findiff
- <https://maroba.github.io/findiff/>
- <https://github.com/maroba/findiff>

```bash
uv pip install findiff
```


In [ ]:
#!uv pip install findiff

In [ ]:
%matplotlib inline
#!/usr/bin/env python3

from findiff import Diff, Id, PDE, FinDiff, BoundaryConditions
import numpy as np


shape = (100, 100)
x, y = np.linspace(0, 1, shape[0]), np.linspace(0, 1, shape[1])
dx, dy = x[1]-x[0], y[1]-y[0]
X, Y = np.meshgrid(x, y, indexing='ij')

L = FinDiff(0, dx, 2) + FinDiff(1, dy, 2)
f = np.zeros(shape)

bc = BoundaryConditions(shape)
bc[1,:] = FinDiff(0, dx, 1), 0  # Neumann BC
bc[-1,:] = 300. - 200*Y   # Dirichlet BC
bc[:, 0] = 300.   # Dirichlet BC
bc[1:-1, -1] = FinDiff(1, dy, 1), 0  # Neumann BC

pde = PDE(L, f, bc)
u = pde.solve()



In [ ]:
import matplotlib.pyplot as plt
plt.imshow(u)

### Py-pde
- <https://py-pde.readthedocs.io/>
- <https://github.com/zwicker-group/py-pde>

```bash
uv pip install py-pde
```


In [ ]:
#!uv pip install py-pde

In [ ]:
import pde

grid = pde.UnitGrid([64, 64])                 # generate grid
state = pde.ScalarField.random_uniform(grid)  # generate initial condition

eq = pde.DiffusionPDE(diffusivity=0.1)        # define the pde
result = eq.solve(state, t_range=10)          # solve the pde
result.plot()                                 # plot the resulting field

In [ ]:
import numpy as np

from pde import CartesianGrid, solve_laplace_equation

grid = CartesianGrid([[0, 2 * np.pi], [0, 2 * np.pi]], 64)
bcs = {"x": {"value": "sin(y)"}, "y": {"value": "sin(x)"}}

res = solve_laplace_equation(grid, bcs)
res.plot()

In [ ]:
import pde

grid = pde.UnitGrid([32, 32])  # generate grid
state = pde.ScalarField.random_uniform(grid)  # generate initial condition

storage = pde.MemoryStorage()

trackers = [
    "progress",  # show progress bar during simulation
    "steady_state",  # abort when steady state is reached
    storage.tracker(interrupts=1),  # store data every simulation time unit
    pde.PlotTracker(show=True),  # show images during simulation
    # print some output every 5 real seconds:
    pde.PrintTracker(interrupts=pde.RealtimeInterrupts(duration=5)),
]

eq = pde.DiffusionPDE(0.1)  # define the PDE
eq.solve(state, 3, dt=0.1, tracker=trackers)

for field in storage:
    print(field.integral)


### Devito
- <https://www.devitoproject.org/>
- <https://www.devitoproject.org/download.html>

In [ ]:
#!pip install devito # better to do it in a different env

## Finite Element Method libs

Introduction to the FEM: <https://www.youtube.com/watch?v=GHjopp47vvQ>

### Scikit-fem
- <https://scikit-fem.readthedocs.io/en/latest/>
- <https://github.com/kinnala/scikit-fem>


In [ ]:
# !uv pip install pip
# !pip install scikit-fem

In [ ]:
# from: https://github.com/kinnala/scikit-fem/blob/master/docs/examples/ex01.py
from skfem import *
from skfem.helpers import dot, grad

# # enable additional mesh validity checks, sacrificing performance
# import logging
# logging.basicConfig(format='%(levelname)s %(asctime)s %(name)s %(message)s')
# logging.getLogger('skfem').setLevel(logging.DEBUG)

# create the mesh
m = MeshTri().refined(6)
# or, with your own points and cells:
# m = MeshTri(points, cells)
# or, load from file
# m = MeshTri.load("mesh.msh")

e = ElementTriP1()
basis = Basis(m, e)


@BilinearForm
def laplace(u, v, _):
    return dot(grad(u), grad(v))


@LinearForm
def rhs(v, _):
    return 1.0 * v


A = laplace.assemble(basis)
b = rhs.assemble(basis)

# enforce Dirichlet boundary conditions
A, b = enforce(A, b, D=m.boundary_nodes())

# solve -- can be anything that takes a sparse matrix and a right-hand side
x = solve(A, b)

def visualize():
    from skfem.visuals.matplotlib import plot
    return plot(m, x, shading='gouraud', colorbar=True)

if __name__ == "__main__":
    visualize().show()

### MFEM

Although it is c++, it has a python wrapper

- <https://mfem.org/>
- <https://github.com/mfem/PyMFEM>
- <https://mfem.org/examples/>
- State of MFEM 2025: <https://www.youtube.com/watch?v=Sq1EwYyjFzc>
- es in HPC: <https://www.youtube.com/watch?v=GHjopp47vvQ>
- <https://colab.research.google.com/github/mfem/pymfem/blob/master/examples/jupyter/ex1.ipynb>
- <https://colab.research.google.com/github/mfem/pymfem/blob/master/examples/jupyter/ex9.ipynb>
- <https://github.com/mfem/PyMFEM/tree/master/examples>


Only works on linux 
```bash
uv pip install pip
pip install mfem # be sure to be used the uv installed pip
```

(for advanced options, like parallelization, you should download the sources and compile with the right options)

In [ ]:
#!uv pip install pip
#!pip install mfem # be sure to be used the uv installed pip. NOTE: TAKES A LOT OF TIME BUILDING
#!uv pip install mpi4py

In [ ]:
# Poisson equation
import mfem.ser as mfem

# --------------------------------------------------
# Mesh
# --------------------------------------------------
mesh = mfem.Mesh.MakeCartesian2D(
    10, 10,
    mfem.Element.QUADRILATERAL,
    True,
    1.0, 1.0
)

# --------------------------------------------------
# Finite element space
# --------------------------------------------------
fec = mfem.H1_FECollection(1, mesh.Dimension())
fes = mfem.FiniteElementSpace(mesh, fec)

print("Number of unknowns:", fes.GetTrueVSize())

# --------------------------------------------------
# Essential boundary conditions
# --------------------------------------------------
ess_bdr = mfem.intArray(mesh.bdr_attributes.Max())
ess_bdr.Assign(1)

ess_tdof_list = mfem.intArray()
fes.GetEssentialTrueDofs(ess_bdr, ess_tdof_list)

# --------------------------------------------------
# Right-hand side
# --------------------------------------------------
one = mfem.ConstantCoefficient(1.0)

b = mfem.LinearForm(fes)
b.AddDomainIntegrator(mfem.DomainLFIntegrator(one))
b.Assemble()

# --------------------------------------------------
# Bilinear form
# --------------------------------------------------
a = mfem.BilinearForm(fes)
a.AddDomainIntegrator(mfem.DiffusionIntegrator())
a.Assemble()

# --------------------------------------------------
# Linear system
# --------------------------------------------------
x = mfem.GridFunction(fes)
x.Assign(0.0)

A = mfem.OperatorPtr()
B = mfem.Vector()
X = mfem.Vector()

a.FormLinearSystem(
    ess_tdof_list,
    x,
    b,
    A,
    X,
    B
)

# --------------------------------------------------
# Solve
# --------------------------------------------------
solver = mfem.GSSmoother(A.AsSparseMatrix())
mfem.PCG(
    A,
    solver,
    B,
    X,
    1,
    200,
    1e-12,
    0.0
)

# Recover finite element solution
a.RecoverFEMSolution(X, b, x)

# Save solution
mesh.Print("mesh.mesh", 8)
x.Save("solution.gf", 8)

print("Done.")

### Fenics
- <https://fenicsproject.org/
- <https://fenicsproject.org/documentation/
- <https://docs.fenicsproject.org/dolfinx/v0.9.0/python/demos.html
- <https://jsdokken.com/dolfinx-tutorial/chapter1/fundamentals.html
- <https://docs.pyvista.org/

Installation is much more complicated than <https://fenicsproject.org/documentation/>
```bash
pip install fenics dolphin mpi4py
```

It is better to use a docker container or use the binder setups
```bash
docker run -ti dolfinx/dolfinx:stable
```


See below for an example using pixi dev.

Example: 2D poisson equation

- Notation and method intro: https://jsdokken.com/dolfinx-tutorial/chapter1/fundamentals.html
- Implementation (use binder) : https://jsdokken.com/dolfinx-tutorial/chapter1/fundamentals_code.html

<img src="https://jsdokken.com/dolfinx-tutorial/_images/u_time.gif" alt="example" width="100%" align="center">

From : <https://jsdokken.com/dolfinx-tutorial/chapter2/diffusion_code.html>




#### Example using pixi dev
According to <https://fenicsproject.org/documentation/>, the recommended way to use fenics (the latest version) is to use conda. But we can do even better by using <https://pixi.prefix.dev/latest/>. This allows to get the full conda repo with uv support when needed, plus many other things. Do the following in your machine

- [ ] Install pixi:      
  ```bash
  curl -fsSL https://pixi.sh/install.sh | sh
  ```
  It will install pixi on `~/.local/bin`. You might need to close and reopen the shell.
- [ ] Create a virtual pixi env with conda-forge as repo, and enter 
  ```bash
  pixi init fenicsx-env -c conda-forge -c nodefaults
  cd fenicsx-en
  ```
- [ ] Add and install the needed packages
  ```bash
  pixi add fenics-dolfinx mpich pyvista
  ```
- [ ] Run the scripts through pixi
  ```bash
  pixi run python your_script.py
  ```
      

In [ ]:
%%writefile fenics_example_01.py
from mpi4py import MPI
from dolfinx import mesh, fem
from dolfinx.fem.petsc import LinearProblem
import ufl
import numpy as np

domain = mesh.create_unit_square(MPI.COMM_WORLD, 32, 32)
V = fem.functionspace(domain, ("Lagrange", 1))

print(f"Mesh: {domain.topology.index_map(domain.topology.dim).size_local} cells, "
      f"{V.dofmap.index_map.size_local} DOFs")

def boundary(x):
    return np.full(x.shape[1], True)

facets = mesh.locate_entities_boundary(domain, domain.topology.dim - 1, boundary)
dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, facets)
bc = fem.dirichletbc(0.0, dofs, V)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
f = fem.Constant(domain, -6.0)
a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx

problem = LinearProblem(
    a, L, bcs=[bc],
    petsc_options_prefix="poisson_",
)
uh = problem.solve()

# --- Diagnostics: print solution stats ---
u_array = uh.x.array
print(f"Solution min: {u_array.min():.6f}")
print(f"Solution max: {u_array.max():.6f}")
print(f"Solution mean: {u_array.mean():.6f}")
print(f"First 10 values: {u_array[:10]}")

# --- Visualization with pyvista ---
import pyvista
from dolfinx import plot

pyvista.OFF_SCREEN = True  # set False if you want an interactive window

topology, cell_types, geometry = plot.vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
grid.point_data["u"] = u_array
grid.set_active_scalars("u")

plotter = pyvista.Plotter()
plotter.add_mesh(grid, show_edges=True, cmap="viridis")
plotter.view_xy()
plotter.add_scalar_bar("u")
plotter.screenshot("solution.png")
print("Saved plot to solution.png")

# Optional: interactive window if not off-screen
# plotter.show()


In [ ]:
%%writefile fenics_example_02.py
# From https://github.com/FEniCS/dolfinx/blob/main/python/demo/demo_po
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.13.6
# ---

# # Poisson equation
#
# This demo illustrates how to:
#
# - Create a {py:class}`function space <dolfinx.fem.FunctionSpace>`
# - Solve a linear partial differential equation
#
# ```{admonition} Download sources
# :class: download
# * {download}`Python script <./demo_poisson.py>`
# * {download}`Jupyter notebook <./demo_poisson.ipynb>`
# ```
# ## Equation and problem definition
#
# For a domain $\Omega \subset \mathbb{R}^n$ with boundary $\partial
# \Omega = \Gamma_{D} \cup \Gamma_{N}$, the Poisson equation with
# particular boundary conditions reads:
#
# $$
# \begin{aligned}
#   - \nabla^{2} u &= f \quad {\rm in} \ \Omega, \\
#   u &= 0 \quad {\rm on} \ \Gamma_{D}, \\
#   \nabla u \cdot n &= g \quad {\rm on} \ \Gamma_{N}. \\
# \end{aligned}
# $$
#
# where $f$ and $g$ are input data and $n$ denotes the outward directed
# boundary normal. The variational problem reads: find $u \in V$ such
# that
#
# $$
# a(u, v) = L(v) \quad \forall \ v \in V,
# $$
#
# where $V$ is a suitable function space and
#
# $$
# \begin{aligned}
#   a(u, v) &:= \int_{\Omega} \nabla u \cdot \nabla v \, {\rm d} x, \\
#   L(v)    &:= \int_{\Omega} f v \, {\rm d} x + \int_{\Gamma_{N}} g v \,
#               {\rm d} s.
# \end{aligned}
# $$
#
# The expression $a(u, v)$ is the bilinear form and $L(v)$
# is the linear form. It is assumed that all functions in $V$
# satisfy the Dirichlet boundary conditions ($u = 0 \ {\rm on} \
# \Gamma_{D}$).
#
# In this demo we consider:
#
# - $\Omega = [0,2] \times [0,1]$ (a rectangle)
# - $\Gamma_{D} = \{(0, y) \cup (2, y) \subset \partial \Omega\}$
# - $\Gamma_{N} = \{(x, 0) \cup (x, 1) \subset \partial \Omega\}$
# - $g = \sin(5x)$
# - $f = 10\exp(-((x - 0.5)^2 + (y - 0.5)^2) / 0.02)$
#
# ## Implementation
#
# The modules that will be used are imported:

# +
from pathlib import Path

from mpi4py import MPI
from petsc4py.PETSc import ScalarType  # type: ignore

import numpy as np

import ufl
from dolfinx import fem, io, mesh, plot
from dolfinx.fem.petsc import LinearProblem

# -

# Note that it is important to first `from mpi4py import MPI` to
# ensure that MPI is correctly initialised.

# We create a rectangular {py:class}`Mesh <dolfinx.mesh.Mesh>` using
# {py:func}`create_rectangle <dolfinx.mesh.create_rectangle>`, and
# create a finite element {py:class}`function space
# <dolfinx.fem.FunctionSpace>` $V$ on the mesh.

# +
msh = mesh.create_rectangle(
    comm=MPI.COMM_WORLD,
    points=((0.0, 0.0), (2.0, 1.0)),
    n=(32, 16),
    cell_type=mesh.CellType.triangle,
)
V = fem.functionspace(msh, ("Lagrange", 1))
# -

# The second argument to {py:func}`functionspace
# <dolfinx.fem.functionspace>` is a tuple `(family, degree)`, where
# `family` is the finite element family, and `degree` specifies the
# polynomial degree. In this case `V` is a space of continuous Lagrange
# finite elements of degree 1. For further details of how one can specify
# finite elements as tuples, see {py:class}`ElementMetaData
# <dolfinx.fem.ElementMetaData>`.
#
# To apply the Dirichlet boundary conditions, we find the mesh facets
# (entities of topological co-dimension 1) that lie on the boundary
# $\Gamma_D$ using {py:func}`locate_entities_boundary
# <dolfinx.mesh.locate_entities_boundary>`. The function is provided
# with a 'marker' function that returns `True` for points `x` on the
# boundary and `False` otherwise.

tdim = msh.topology.dim
fdim = tdim - 1
facets = mesh.locate_entities_boundary(
    msh,
    dim=fdim,
    marker=lambda x: np.isclose(x[0], 0.0) | np.isclose(x[0], 2.0),
)

# We now find the degrees-of-freedom that are associated with the
# boundary facets using {py:func}`locate_dofs_topological
# <dolfinx.fem.locate_dofs_topological>`:

dofs = fem.locate_dofs_topological(V=V, entity_dim=fdim, entities=facets)

# and use {py:func}`dirichletbc <dolfinx.fem.dirichletbc>` to create a
# {py:class}`DirichletBC <dolfinx.fem.DirichletBC>` class that
# represents the boundary condition:

bc = fem.dirichletbc(value=ScalarType(0), dofs=dofs, V=V)

# Next, the variational problem is defined:

# +
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
x = ufl.SpatialCoordinate(msh)
f = 10 * ufl.exp(-((x[0] - 0.5) ** 2 + (x[1] - 0.5) ** 2) / 0.02)
g = ufl.sin(5 * x[0])
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = ufl.inner(f, v) * ufl.dx + ufl.inner(g, v) * ufl.ds
# -

# A {py:class}`LinearProblem <dolfinx.fem.petsc.LinearProblem>` object is
# created that brings together the variational problem, the Dirichlet
# boundary condition, and which specifies the linear solver. In this
# case an LU solver is used, and we ask that PETSc throws an error
# if the solver does not converge. The {py:func}`solve
# <dolfinx.fem.petsc.LinearProblem.solve>` computes the solution.

# +
problem = LinearProblem(
    a,
    L,
    bcs=[bc],
    petsc_options_prefix="demo_poisson_",
    petsc_options={"ksp_type": "preonly", "pc_type": "lu", "ksp_error_if_not_converged": True},
)
uh = problem.solve()
assert isinstance(uh, fem.Function)
# -

# The solution can be written to a {py:class}`XDMFFile
# <dolfinx.io.XDMFFile>` file visualization with [ParaView](https://www.paraview.org/)
# or [VisIt](https://visit-dav.github.io/visit-website/):

# +
out_folder = Path("out_poisson")
out_folder.mkdir(parents=True, exist_ok=True)
with io.XDMFFile(msh.comm, out_folder / "poisson.xdmf", "w") as file:
    file.write_mesh(msh)
    file.write_function(uh)
# -

# and displayed using [pyvista](https://docs.pyvista.org/).

# +
try:
    import pyvista

    cells, types, x = plot.vtk_mesh(V)
    grid = pyvista.UnstructuredGrid(cells, types, x)
    grid.point_data["u"] = uh.x.array.real
    grid.set_active_scalars("u")
    plotter = pyvista.Plotter()
    plotter.add_mesh(grid, show_edges=True)
    warped = grid.warp_by_scalar()
    plotter.add_mesh(warped)
    if pyvista.OFF_SCREEN:
        plotter.screenshot(out_folder / "uh_poisson.png")
    else:
        plotter.show()
except ModuleNotFoundError:
    print("'pyvista' is required to visualise the solution.")
    print("To install pyvista with pip: 'python3 -m pip install pyvista'.")
# -

## To Check
- https://getfem-examples.readthedocs.io/en/latest/demo_unit_disk.html
- https://www.pygimli.org/_tutorials_auto/2_modelling/plot_2-mod-fem.html
- https://flocode.substack.com/p/049-pynite-01-introduction-to-finite
- https://bleyerj.github.io/comet-fenicsx/
- https://gmshmodel.readthedocs.io/en/latest/